# Extracting building polygons

Following the `Geo_Inference` tutorial, we would like to now convert the irregular extracted masks into regularized building polygons.

## Preparing the Python environment

To start, we need to install the geo-inference package in our environment. We assume we already built a "tutorial" environment using the requirements files available on the root of this repository.
- [If you have GPU available on your device](https://github.com/geoaiclassroom/geoai_learning/blob/main/requirements_gpu.txt)
- [If you only have CPU available on your device](https://github.com/geoaiclassroom/geoai_learning/blob/main/requirements_cpu.txt)

Then, we should add geo-inference to this environment using the same instructions a the `Geo_Inference` tutorial.

Then, activate your "tutorial" environment (*conda activate tutorial*) and install `buildingregulariser` package. 

Then, install the package:

    pip install buildingregulariser==0.2.4 
    pip install --force-reinstall numpy==1.26.4


In [1]:
# Run this cell to fix "RuntimeError: This event loop is already running" error when running inference in the notebook. This is a known issue with Jupyter notebooks and async code, and nest_asyncio is a common workaround.
# Also, due to some conflicts with our rasterio/gdal versions, the order of importing torch versus gdal matters - so we would on top import torch to avoid dyamic library conflicts.
import torch
import nest_asyncio
nest_asyncio.apply()

In [2]:
# Run this cell as a monkeypatch if you are running this tutorial on a Windows machine.

# Workaround for Windows file-locking during rioxarray+dask chunked writes.
# geo_inference passes a lock, which makes rioxarray writes by reopening the same output TIFF in r+ per chunk; 
# this operation can fail on Windows with an error expressing "file used by other process".

import rioxarray.raster_array as rxr_raster_array

_original_to_raster = rxr_raster_array.RasterArray.to_raster

def _to_raster_windows_safe(self, *args, **kwargs):
    kwargs["lock"] = False
    kwargs["windowed"] = False
    return _original_to_raster(self, *args, **kwargs)

rxr_raster_array.RasterArray.to_raster = _to_raster_windows_safe
print("Applied Windows-safe rioxarray to_raster patch (lock=False, windowed=False).")

Applied Windows-safe rioxarray to_raster patch (lock=False, windowed=False).


In [3]:
import zipfile
from pathlib import Path
from datetime import datetime
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import rasterio
import geopandas as gpd
import os
from buildingregulariser import regularize_geodataframe
from shapely.geometry import Polygon, MultiPolygon
from geo_inference.geo_inference import GeoInference

In [4]:
zip_path = Path("./Data/waterloo_crop_2020.zip")
extract_dir = zip_path.with_suffix("")  # removes .zip

# Unzip only if not already done
if not extract_dir.exists():
    print(f"Extracting to {extract_dir}...")
    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(extract_dir)
else:
    print(f"{extract_dir} already extracted.")

Data\waterloo_crop_2020 already extracted.


## Running Inference
We could either run the inference on our geotif file using command line arguments or by setting everything in a config file. The geo-inference repo provides you with details on how a config file could be set. For clarity, in this tutorial, we will set every thing as arguments.

In [5]:
# Unique output directory per run to avoid path collisions
base_work_dir = Path(r"./Data/waterloo_crop_2020")
run_work_dir = base_work_dir / f"run_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
run_work_dir.mkdir(parents=True, exist_ok=True)

# Initialize the GeoInference object
geo_inf = GeoInference(
    model=r"./logs/mlrun/scripted.pt",
    work_dir=str(run_work_dir),
    mask_to_vec=True,
    mask_to_yolo=False,
    mask_to_coco=False, 
    device="gpu",
    multi_gpu=False,
    gpu_id=0, 
    num_classes=2,
    prediction_threshold=0.5,
    transformers=True,
    transformer_flip=False,
    transformer_rotate=True,
)


In [ ]:
# Perform feature extraction on a TIFF image
image_path = r"./Data/waterloo_crop_2020/sample.tif"
bands_requested = [1, 2, 3] # RGB bands
workers = 1
patch_size = 512
# bbox = "0, 0, 1000, 1000"
geo_inf(
    inference_input = image_path,  
    bands_requested = bands_requested, 
    patch_size = patch_size, 
    workers = workers, 
    bbox=None
)

## Regularizing buildings

- First, we convert the geojson input to a geodataframe
- Then, we perform some basic cleaning; e.g. removing small artifacts and removing holes from inside the polygons
- Finally, we apply the regularizer on the geodataframe and store the results as a geopackage

In [7]:
candidates = sorted(Path(run_work_dir).glob("*_polygons_*.geojson"), key=lambda p: p.stat().st_mtime)
if not candidates:
    raise FileNotFoundError(f"No GeoJson file found in {run_work_dir}. Run inference first.")
geojson_path = candidates[-1]

In [8]:
def remove_small_holes(geom, min_area):
    if geom is None:
        return None

    if geom.geom_type == "Polygon":
        # keep only holes larger than threshold
        new_holes = []
        for hole in geom.interiors:
            hole_poly = Polygon(hole)
            if hole_poly.area >= min_area:
                new_holes.append(hole)

        return Polygon(geom.exterior, new_holes)

    elif geom.geom_type == "MultiPolygon":
        return MultiPolygon([remove_small_holes(p, min_area) for p in geom.geoms])

    else:
        return geom

def vectorize_instances_to_gpkg(input_geojson, output_path, min_building_area):
    
    gdf = gpd.read_file(input_geojson)
    gdf = gdf[gdf.geometry.area >= min_building_area]
    
    gdf["geometry"] = gdf["geometry"].apply(
        lambda g: remove_small_holes(g, min_building_area)
    )

    regularized = regularize_geodataframe(
        gdf,
        parallel_threshold=3,  # Higher values allow less edge alignment
        simplify_tolerance=3,  # Controls simplification level, should be 2-3 x the raster pixel size
        allow_45_degree=True,  # Enable 45-degree angles
        allow_circles=False,  # Enable circle detection
        circle_threshold=0.98,  # IOU threshold for circle detection
        neighbor_alignment = False,  # After regularization try to align each building with neighboring buildings
        neighbor_search_distance= 100.0,  # The search distance around each building to find neighbors
        neighbor_max_rotation=10  # The maximum rotation allowed to align with neighbors
        )

    regularized.to_file(output_path, driver="GPKG")

In [9]:
vectorize_instances_to_gpkg(geojson_path, os.path.join(run_work_dir, "regularized_buildings.gpkg"), min_building_area=200)

[2026-06-04 15:38:43,542][pyogrio._io][INFO] - Created 6 records
